# Importing and vizualizing dataset

In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import pandas as pd
from PIL import Image

from pathlib import Path

In [2]:
filedir = Path('./datasets/')

train_dataframe = pd.read_csv(filedir / 'train_data.csv')
test_dataframe = pd.read_csv(filedir / 'test_data.csv')

In [3]:
train_dataframe.head()

,ID,patient_id,breast density,left or right breast,image view,abnormality id,abnormality type,calc type,calc distribution,assessment,pathology,subtlety
0,Calc-Training_P_00005_RIGHT_CC_1,P_00005,3,RIGHT,CC,1,calcification,AMORPHOUS,CLUSTERED,3,MALIGNANT,3
1,Calc-Training_P_00005_RIGHT_MLO_1,P_00005,3,RIGHT,MLO,1,calcification,AMORPHOUS,CLUSTERED,3,MALIGNANT,3
2,Calc-Training_P_00007_LEFT_CC_1,P_00007,4,LEFT,CC,1,calcification,PLEOMORPHIC,LINEAR,4,BENIGN,4
3,Calc-Training_P_00007_LEFT_MLO_1,P_00007,4,LEFT,MLO,1,calcification,PLEOMORPHIC,LINEAR,4,BENIGN,4
4,Calc-Training_P_00008_LEFT_CC_1,P_00008,1,LEFT,CC,1,calcification,NaN,REGIONAL,2,BENIGN_WITHOUT_CALLBACK,3


# Definition and neural network model design

In [4]:
class CNN(nn.Module):
    def __init__(self, num_classes, additional_features_size=0):
        super(CNN, self).__init__()
        # Convolutional layers
        self.conv_layer1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv_layer2 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1)
        self.max_pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv_layer3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.conv_layer4 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1)
        self.max_pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Fully connected layers
        self.flattened_size = None  # To be calculated dynamically
        self.fc1 = None  # To be initialized dynamically
        self.relu = nn.ReLU()
        
        # Additional features
        self.additional_features_size = additional_features_size
        if self.additional_features_size > 0:
            self.fc_additional = nn.Linear(self.additional_features_size, 64)
        
        # Final classification layer
        self.fc_final = nn.Linear(128 if additional_features_size > 0 else 64, num_classes)
        
    def forward(self, x, additional_features=None):
        # Convolutional layers with ReLU activation
        x = self.relu(self.conv_layer1(x))
        x = self.relu(self.conv_layer2(x))
        x = self.max_pool1(x)
        
        x = self.relu(self.conv_layer3(x))
        x = self.relu(self.conv_layer4(x))
        x = self.max_pool2(x)
        
        # Flatten dynamically
        x = x.view(x.size(0), -1)
        
        # Initialize fc1 if not done (dynamic size)
        if self.fc1 is None:
            self.flattened_size = x.size(1)
            self.fc1 = nn.Linear(self.flattened_size, 64)
            self.fc1.to(x.device)
        
        x = self.relu(self.fc1(x))
        
        # Add additional features if provided
        if additional_features is not None and self.additional_features_size > 0:
            additional_features = self.relu(self.fc_additional(additional_features))
            x = torch.cat((x, additional_features), dim=1)
        
        # Final classification layer
        x = self.fc_final(x)
        return x


### Definition of hyperparameters

In [9]:
batch_size = 64  # We will be batch learining to speed up learning process

num_classes = len(train_dataframe['pathology'].unique())  # There are 3 target classes

num_epochs = 200  # Training period

model = CNN(num_classes)  # Set model parameters

optimizer = torch.optim.Adam(model.parameters())  # Default learning rate 10^-3 which sounds fine

criterion = nn.CrossEntropyLoss()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # Whether to train on GPU (cuda) or CPU

model.to(device)

CNN(
  (conv_layer1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv_layer2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (max_pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv_layer3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv_layer4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (max_pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (relu): ReLU()
  (fc_final): Linear(in_features=64, out_features=3, bias=True)
)

In [10]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])  # Normalize to [-1, 1]
])

In [ ]:
class BreastCancerDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe
        self.image_dir = image_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image_path = f"{self.image_dir}/{row['ID']}.jpg"  # Ensure path matches your structure
        image = Image.open(image_path).convert("L")  # Grayscale image
        
        label = row['pathology']  # Target label
        label = 0 if label == 'BENIGN_WITHOUT_CALLBACK' else 1 if label == 'BENIGN' else 2  # Encoding classes
        
        additional_features = row[['breast density', 'left or right breast', 'image view']]  # Example
        additional_features = torch.tensor(additional_features.values, dtype=torch.float32)
        
        if self.transform:
            image = self.transform(image)
        
        return image, label, additional_features

In [ ]:
from torch.utils.data import DataLoader

train_dataset = BreastCancerDataset(train_dataframe, 'train_cropped_images', transform=transform)
test_dataset = BreastCancerDataset(test_dataframe, 'test_cropped_images', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

### Training loop

In [ ]:
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    all_preds = []
    all_labels = []
    
    for images, labels, additional_features in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        additional_features = additional_features.to(device)
        
        # Forward pass
        outputs = model(images, additional_features)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Accumulate loss and predictions
        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    train_acc = accuracy_score(all_labels, all_preds)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {train_loss/len(train_loader):.4f}, Accuracy: {train_acc:.4f}")
    
    # Validation loop
    model.eval()
    val_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels, additional_features in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            additional_features = additional_features.to(device)
            
            outputs = model(images, additional_features)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    val_acc = accuracy_score(all_labels, all_preds)
    print(f"Validation Loss: {val_loss/len(test_loader):.4f}, Accuracy: {val_acc:.4f}")